In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/demand-forecasting-kernels-only/sample_submission.csv
/kaggle/input/competitions/demand-forecasting-kernels-only/train.csv
/kaggle/input/competitions/demand-forecasting-kernels-only/test.csv


In [2]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import statsmodels.api as sm

pd.set_option('display.max_columns', 50)

# lightgbm and statsmodels ship pre-installed in Kaggle's default Python environment,
# so no pip install / internet access is needed here.


## 2. Load Data

List the input directory first so the path is always correct, even if the competition slug changes.

In [3]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/demand-forecasting-kernels-only/sample_submission.csv
/kaggle/input/competitions/demand-forecasting-kernels-only/train.csv
/kaggle/input/competitions/demand-forecasting-kernels-only/test.csv


In [4]:
INPUT_DIR = '/kaggle/input/competitions/demand-forecasting-kernels-only'

train_full = pd.read_csv(f'{INPUT_DIR}/train.csv', parse_dates=['date'])
test = pd.read_csv(f'{INPUT_DIR}/test.csv', parse_dates=['date'])
sample_submission = pd.read_csv(f'{INPUT_DIR}/sample_submission.csv')

print("Train shape:", train_full.shape)
print("Test shape:", test.shape)
train_full.head()

Train shape: (913000, 4)
Test shape: (45000, 4)


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


## 3. Exploratory Data Analysis

Check date ranges, coverage, and basic stats.

In [5]:
print("TRAIN date range:", train_full['date'].min(), "->", train_full['date'].max())
print("TEST date range:", test['date'].min(), "->", test['date'].max())
print("Stores:", sorted(train_full['store'].unique()))
print("Items:", train_full['item'].nunique())
print("\nMissing values:\n", train_full.isnull().sum())

# every store-item pair should have full daily coverage, no gaps
combo_counts = train_full.groupby(['store','item']).size()
print("\nUnique daily-count values per store-item combo (should be a single number):", combo_counts.unique())


TRAIN date range: 2013-01-01 00:00:00 -> 2017-12-31 00:00:00
TEST date range: 2018-01-01 00:00:00 -> 2018-03-31 00:00:00
Stores: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Items: 50

Missing values:
 date     0
store    0
item     0
sales    0
dtype: int64

Unique daily-count values per store-item combo (should be a single number): [1826]


In [6]:
train_full['year'] = train_full['date'].dt.year
train_full['month'] = train_full['date'].dt.month
train_full['dow'] = train_full['date'].dt.dayofweek

print("=== Yearly total sales (growth trend) ===")
print(train_full.groupby('year')['sales'].sum())

print("\n=== Avg sales by month (seasonality) ===")
print(train_full.groupby('month')['sales'].mean().round(2))

print("\n=== Avg sales by day-of-week (0=Mon) ===")
print(train_full.groupby('dow')['sales'].mean().round(2))

print("\n=== Avg sales by store ===")
print(train_full.groupby('store')['sales'].mean().round(2))

print("\n=== Item mean sales range ===")
item_means = train_full.groupby('item')['sales'].mean()
print(item_means.min(), "-", item_means.max())

=== Yearly total sales (growth trend) ===
year
2013     7941243
2014     9135482
2015     9536887
2016    10357160
2017    10733740
Name: sales, dtype: int64

=== Avg sales by month (seasonality) ===
month
1     35.52
2     39.38
3     47.31
4     55.15
5     59.13
6     63.03
7     67.00
8     59.11
9     55.07
10    51.19
11    55.22
12    39.37
Name: sales, dtype: float64

=== Avg sales by day-of-week (0=Mon) ===
dow
0    41.43
1    48.23
2    48.37
3    51.72
4    55.16
5    58.66
6    62.14
Name: sales, dtype: float64

=== Avg sales by store ===
store
1     47.27
2     67.03
3     59.53
4     54.90
5     39.77
6     39.73
7     36.36
8     64.14
9     55.05
10    58.71
Name: sales, dtype: float64

=== Item mean sales range ===
18.35870755750274 - 88.03077765607885


## 4. Key Discovery: Clean Multiplicative Structure

Check whether the year-over-year growth curve is the *same* across every store and item (i.e. a single global trend, not 500 different trends).

In [7]:
store_year = train_full.groupby(['store','year'])['sales'].mean().unstack()
store_year_norm = store_year.div(store_year[2013], axis=0)
print("Store-level growth normalized to 2013:")
print(store_year_norm.round(3))

item_year = train_full.groupby(['item','year'])['sales'].mean().unstack()
item_year_norm = item_year.div(item_year[2013], axis=0)
print("\nStd of item-level normalized growth across items, per year (near-zero = uniform trend):")
print(item_year_norm.std().round(4))


Store-level growth normalized to 2013:
year   2013   2014   2015   2016   2017
store                                  
1       1.0  1.152  1.200  1.302  1.354
2       1.0  1.149  1.198  1.300  1.349
3       1.0  1.151  1.202  1.303  1.354
4       1.0  1.149  1.201  1.300  1.350
5       1.0  1.152  1.201  1.304  1.353
6       1.0  1.150  1.201  1.297  1.350
7       1.0  1.152  1.203  1.301  1.353
8       1.0  1.151  1.201  1.300  1.352
9       1.0  1.151  1.204  1.302  1.354
10      1.0  1.149  1.199  1.299  1.349

Std of item-level normalized growth across items, per year (near-zero = uniform trend):
year
2013    0.0000
2014    0.0040
2015    0.0044
2016    0.0039
2017    0.0058
dtype: float64


In [8]:
# global_avg(date) = mean sales across all 500 store-item series for a given date
# (idiosyncratic noise cancels out, leaving pure trend+seasonality)
global_pattern = train_full.groupby('date')['sales'].mean().rename('global_avg')
tmp = train_full.merge(global_pattern, on='date')
tmp['ratio'] = tmp['sales'] / tmp['global_avg']

# if sales = store_item_factor * global_pattern, this ratio should be ~constant over time per store-item
si_ratio_stats = tmp.groupby(['store','item'])['ratio'].agg(['mean','std'])
si_ratio_stats['cv'] = si_ratio_stats['std'] / si_ratio_stats['mean']
print("Coefficient of variation of the store-item ratio (mostly noise, not drift):")
print(si_ratio_stats['cv'].describe())

# does the weekly seasonal *shape* differ by store? (check after removing each store's own level)
tmp['dow'] = tmp['date'].dt.dayofweek
dow_by_store = tmp.groupby(['store','dow'])['ratio'].mean().unstack()
dow_by_store_norm = dow_by_store.div(dow_by_store.mean(axis=1), axis=0)
print("\nStd across stores for each day-of-week (near-zero = identical weekly shape everywhere):")
print(dow_by_store_norm.std().round(4))


Coefficient of variation of the store-item ratio (mostly noise, not drift):
count    500.000000
mean       0.156447
std        0.041369
min        0.096008
25%        0.122980
50%        0.146004
75%        0.183964
max        0.294968
Name: cv, dtype: float64

Std across stores for each day-of-week (near-zero = identical weekly shape everywhere):
dow
0    0.0012
1    0.0008
2    0.0015
3    0.0009
4    0.0015
5    0.0008
6    0.0011
dtype: float64


**Conclusion:** the growth trend and weekly seasonality are essentially identical across all 500 store-item series. This means `sales(store, item, date) ≈ store_item_level × global_trend_and_seasonality(date)`, which we can exploit directly for a robust, extrapolation-safe trend model.

## 5. Decomposition Model

Fit a log-linear regression on the daily average series:

`log(global_avg) = trend(day_index) + weekly dummies + annual Fourier seasonality`

This is safe to extrapolate into 2018 because the trend term is linear (no runaway curvature) and the Fourier terms are periodic by construction.

In [9]:
def build_design_matrix(dates, day_index, n_harmonics=4):
    """Trend + day-of-week dummies + Fourier annual seasonality features."""
    df = pd.DataFrame({'day_index': day_index})
    dow = pd.Series(dates).dt.dayofweek.values
    for d in range(1, 7):  # Monday(0) as reference category
        df[f'dow_{d}'] = (dow == d).astype(int)
    doy_frac = (day_index % 365.25) / 365.25
    for k in range(1, n_harmonics + 1):
        df[f'sin_{k}'] = np.sin(2 * np.pi * k * doy_frac)
        df[f'cos_{k}'] = np.cos(2 * np.pi * k * doy_frac)
    df = sm.add_constant(df)
    return df


def fit_global_pattern(train_df, min_date):
    """Fit log-linear trend+seasonality model on the daily average across all store-items."""
    gp = train_df.groupby('date')['sales'].mean().rename('global_avg').reset_index()
    gp['day_index'] = (gp['date'] - min_date).dt.days
    X = build_design_matrix(gp['date'], gp['day_index'])
    y = np.log(gp['global_avg'])
    model = sm.OLS(y, X).fit()
    return model


def predict_global_pattern(model, min_date, dates):
    day_index = (pd.to_datetime(dates) - min_date).dt.days
    X = build_design_matrix(pd.to_datetime(dates), day_index)
    log_pred = model.predict(X)
    return np.exp(log_pred)


def fit_store_item_factors(train_df):
    gp = train_df.groupby('date')['sales'].mean().rename('global_avg')
    tmp = train_df.merge(gp, on='date')
    tmp['ratio'] = tmp['sales'] / tmp['global_avg']
    factors = tmp.groupby(['store', 'item'])['ratio'].mean()
    return factors


def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1, denom)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)


## 6. Feature Engineering

One function builds every feature needed by both models, including the decomposition prediction itself as a feature for Model B.

In [10]:
MIN_DATE = train_full['date'].min()

def add_all_features(df, decomp_model, si_factors, min_date=MIN_DATE):
    df = df.copy()
    df['dow'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['year'] = df['date'].dt.year
    df['quarter'] = df['date'].dt.quarter
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['dayofyear'] = df['date'].dt.dayofyear
    df['day_index'] = (df['date'] - min_date).dt.days
    df['global_pred'] = predict_global_pattern(decomp_model, min_date, df['date'])
    df = df.merge(si_factors.rename('si_factor'), on=['store', 'item'], how='left')
    df['decomp_pred'] = df['si_factor'] * df['global_pred']
    return df

FEATURES_A = ['store','item','dow','month','day','year','dayofyear','day_index']
CAT_A = ['store','item','dow','month','year']

FEATURES_B = ['store','item','dow','month','day','quarter','weekofyear','dayofyear','si_factor','decomp_pred']
CAT_B = ['store','item','dow','month','quarter']

LGB_PARAMS = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.02,
    'num_leaves': 63,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
}


## 7. Validation

Hold out Jan–Mar of the **last training year** (2017) as a proxy for the real Jan–Mar 2018 test window — training only on data through Dec 2016. This mimics the real forecast horizon (predicting an unseen future year) as closely as possible.

In [11]:
val_cutoff = pd.Timestamp('2017-01-01')
val_end = pd.Timestamp('2017-03-31')
tr = train_full[train_full['date'] < val_cutoff].copy()
val = train_full[(train_full['date'] >= val_cutoff) & (train_full['date'] <= val_end)].copy()

decomp_model_val = fit_global_pattern(tr, MIN_DATE)
si_factors_val = fit_store_item_factors(tr)

tr_f = add_all_features(tr, decomp_model_val, si_factors_val)
val_f = add_all_features(val, decomp_model_val, si_factors_val)

print("R-squared of decomposition fit:", decomp_model_val.rsquared)


R-squared of decomposition fit: 0.9554005541488344


In [12]:
# internal dev split (inside tr) for early stopping, kept separate from the val holdout above
dev_cutoff = pd.Timestamp('2016-10-01')
inner_tr = tr_f[tr_f['date'] < dev_cutoff]
inner_dev = tr_f[tr_f['date'] >= dev_cutoff]

# ---- Model A: direct sales regression ----
ds_tr = lgb.Dataset(inner_tr[FEATURES_A], label=inner_tr['sales'], categorical_feature=CAT_A)
ds_dev = lgb.Dataset(inner_dev[FEATURES_A], label=inner_dev['sales'], categorical_feature=CAT_A, reference=ds_tr)
cv_a = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=3000, valid_sets=[ds_dev],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
print("Model A best_iteration:", cv_a.best_iteration)

full_a = lgb.Dataset(tr_f[FEATURES_A], label=tr_f['sales'], categorical_feature=CAT_A)
gbm_a_val = lgb.train(LGB_PARAMS, full_a, num_boost_round=cv_a.best_iteration)
pred_a_val = gbm_a_val.predict(val_f[FEATURES_A])


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
Model A best_iteration: 627
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero


In [13]:
# ---- Model B: decomposition + LightGBM residual (log ratio target) ----
tr_f['target_b'] = np.log(tr_f['sales'].clip(lower=1) / tr_f['decomp_pred'])
inner_tr_b = tr_f[tr_f['date'] < dev_cutoff]
inner_dev_b = tr_f[tr_f['date'] >= dev_cutoff]

ds_tr_b = lgb.Dataset(inner_tr_b[FEATURES_B], label=inner_tr_b['target_b'], categorical_feature=CAT_B)
ds_dev_b = lgb.Dataset(inner_dev_b[FEATURES_B], label=inner_dev_b['target_b'], categorical_feature=CAT_B, reference=ds_tr_b)
cv_b = lgb.train(LGB_PARAMS, ds_tr_b, num_boost_round=3000, valid_sets=[ds_dev_b],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
print("Model B best_iteration:", cv_b.best_iteration)

full_b = lgb.Dataset(tr_f[FEATURES_B], label=tr_f['target_b'], categorical_feature=CAT_B)
gbm_b_val = lgb.train(LGB_PARAMS, full_b, num_boost_round=cv_b.best_iteration)
pred_b_val = val_f['decomp_pred'].values * np.exp(gbm_b_val.predict(val_f[FEATURES_B]))


Model B best_iteration: 322


In [14]:
print("Model A (direct LightGBM) SMAPE:      ", round(smape(val_f['sales'], pred_a_val), 4))
print("Model B (decomposition hybrid) SMAPE:   ", round(smape(val_f['sales'], pred_b_val), 4))

# naive seasonal baseline for reference: same period, prior year
naive = tr[(tr['date'] >= '2016-01-01') & (tr['date'] <= '2016-03-31')].copy()
naive['date'] = naive['date'] + pd.DateOffset(years=1)
val_naive = val.merge(naive[['store','item','date','sales']], on=['store','item','date'], suffixes=('','_naive'))
print("Naive seasonal baseline SMAPE:          ", round(smape(val_naive['sales'], val_naive['sales_naive']), 4))

best_w, best_score = None, None
for w in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    blend = w * pred_a_val + (1 - w) * pred_b_val
    score = smape(val_f['sales'], blend)
    print(f"Blend w={w} (A) / {1-w:.1f} (B): SMAPE={score:.4f}")
    if best_score is None or score < best_score:
        best_score, best_w = score, w

print(f"\nBest blend weight: {best_w}  ->  SMAPE={best_score:.4f}")


Model A (direct LightGBM) SMAPE:       13.4899
Model B (decomposition hybrid) SMAPE:    13.9317
Naive seasonal baseline SMAPE:           24.3241
Blend w=0.5 (A) / 0.5 (B): SMAPE=13.5378
Blend w=0.6 (A) / 0.4 (B): SMAPE=13.4990
Blend w=0.7 (A) / 0.3 (B): SMAPE=13.4748
Blend w=0.8 (A) / 0.2 (B): SMAPE=13.4654
Blend w=0.9 (A) / 0.1 (B): SMAPE=13.4702
Blend w=1.0 (A) / 0.0 (B): SMAPE=13.4899

Best blend weight: 0.8  ->  SMAPE=13.4654


**Validated result:** the 80/20 blend of Model A and Model B scores **~13.47 SMAPE** on this holdout — a solid, well-validated estimate of real leaderboard performance.

## 8. Final Training on Full Data

Refit the decomposition and both LightGBM models on **all** of 2013–2017, then predict the real Jan–Mar 2018 test period.

In [15]:
decomp_model_full = fit_global_pattern(train_full, MIN_DATE)
si_factors_full = fit_store_item_factors(train_full)
print("Global pattern R-squared (full data):", decomp_model_full.rsquared)

train_f = add_all_features(train_full, decomp_model_full, si_factors_full)
test_f = add_all_features(test, decomp_model_full, si_factors_full)


Global pattern R-squared (full data): 0.9547758728597138


In [16]:
# internal dev split (last ~3 months of full train) for early stopping
dev_cutoff_full = pd.Timestamp('2017-10-01')
inner_tr_full = train_f[train_f['date'] < dev_cutoff_full]
inner_dev_full = train_f[train_f['date'] >= dev_cutoff_full]

# ---- Model A ----
ds_tr = lgb.Dataset(inner_tr_full[FEATURES_A], label=inner_tr_full['sales'], categorical_feature=CAT_A)
ds_dev = lgb.Dataset(inner_dev_full[FEATURES_A], label=inner_dev_full['sales'], categorical_feature=CAT_A, reference=ds_tr)
cv_a_full = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=3000, valid_sets=[ds_dev],
                       callbacks=[lgb.early_stopping(100, verbose=False)])
print("Model A best_iteration (full data):", cv_a_full.best_iteration)

full_train_a = lgb.Dataset(train_f[FEATURES_A], label=train_f['sales'], categorical_feature=CAT_A)
gbm_a_final = lgb.train(LGB_PARAMS, full_train_a, num_boost_round=cv_a_full.best_iteration)


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
Model A best_iteration (full data): 556
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero


In [17]:
# ---- Model B ----
train_f['target_b'] = np.log(train_f['sales'].clip(lower=1) / train_f['decomp_pred'])
inner_tr_b_full = train_f[train_f['date'] < dev_cutoff_full]
inner_dev_b_full = train_f[train_f['date'] >= dev_cutoff_full]

ds_tr_b = lgb.Dataset(inner_tr_b_full[FEATURES_B], label=inner_tr_b_full['target_b'], categorical_feature=CAT_B)
ds_dev_b = lgb.Dataset(inner_dev_b_full[FEATURES_B], label=inner_dev_b_full['target_b'], categorical_feature=CAT_B, reference=ds_tr_b)
cv_b_full = lgb.train(LGB_PARAMS, ds_tr_b, num_boost_round=3000, valid_sets=[ds_dev_b],
                       callbacks=[lgb.early_stopping(100, verbose=False)])
print("Model B best_iteration (full data):", cv_b_full.best_iteration)

full_train_b = lgb.Dataset(train_f[FEATURES_B], label=train_f['target_b'], categorical_feature=CAT_B)
gbm_b_final = lgb.train(LGB_PARAMS, full_train_b, num_boost_round=cv_b_full.best_iteration)


Model B best_iteration (full data): 164


In [18]:
BEST_BLEND_WEIGHT = 0.8  # from validation search above (Model A weight)

pred_a_test = gbm_a_final.predict(test_f[FEATURES_A])
pred_b_test = test_f['decomp_pred'].values * np.exp(gbm_b_final.predict(test_f[FEATURES_B]))

final_pred = BEST_BLEND_WEIGHT * pred_a_test + (1 - BEST_BLEND_WEIGHT) * pred_b_test
final_pred = np.clip(final_pred, 0, None)  # sales can't be negative

test_f['sales'] = final_pred
print(pd.Series(final_pred).describe())


count    45000.000000
mean        47.323094
std         23.754243
min          8.085021
25%         27.615905
50%         43.604015
75%         63.329199
max        137.028895
dtype: float64


## 9. Build & Sanity-Check the Submission

In [19]:
submission = test_f[['id', 'sales']].copy()
submission['id'] = submission['id'].astype(int)

# sanity checks
print("Columns match sample_submission:", list(submission.columns) == list(sample_submission.columns))
print("Row count matches:", len(submission) == len(sample_submission))
print("IDs match exactly (same order):", (submission['id'].values == sample_submission['id'].values).all())
print("Any nulls:", submission.isnull().sum().sum())
print("Any negative predictions:", (submission['sales'] < 0).sum())

jan_mar_2017_actual = train_full[(train_full['date'] >= '2017-01-01') & (train_full['date'] <= '2017-03-31')]['sales'].mean()
print("\nActual Jan-Mar 2017 mean sales:", round(jan_mar_2017_actual, 2))
print("Predicted Jan-Mar 2018 mean sales:", round(submission['sales'].mean(), 2))
print("Implied YoY growth:", round(submission['sales'].mean() / jan_mar_2017_actual - 1, 4))

submission.head(10)


Columns match sample_submission: True
Row count matches: True
IDs match exactly (same order): True
Any nulls: 0
Any negative predictions: 0

Actual Jan-Mar 2017 mean sales: 45.81
Predicted Jan-Mar 2018 mean sales: 47.32
Implied YoY growth: 0.0331


,id,sales
0,0,12.118128
1,1,14.391701
2,2,14.580264
3,3,15.444394
4,4,16.749761
5,5,17.578771
6,6,18.674048
7,7,12.166980
8,8,14.435028
9,9,14.550994


In [20]:
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv, shape:", submission.shape)


Saved submission.csv, shape: (45000, 2)


## Summary

- **Validated SMAPE:** ~13.47 (Jan–Mar 2017 holdout, trained on data through Dec 2016 only)
- **Key insight:** the dataset decomposes almost perfectly into a global trend/seasonality curve × a fixed per-store-item multiplier, which made trend extrapolation into 2018 much safer than relying on tree splits alone.
- **Ideas to push the score further:** add CatBoost/XGBoost as extra ensemble members, add lag features at safe lags (≥91 days, to avoid leakage over the 90-day horizon), or tune the blend weight with multiple time-based folds instead of a single holdout.
